In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("ai4privacy/pii-masking-400k")

In [ ]:
train_ds = ds['train'].filter(lambda x: x['language'] == 'es')['source_text']
dev_ds = ds['validation'].filter(lambda x: x['language'] == 'es')['source_text']

In [ ]:
ds['train'].filter(lambda x: x['language'] == 'es')[20]

In [ ]:
for item in train_ds:
    print(item)
    break

In [ ]:
import html
import re
import requests
import unicodedata

API_URL = "http://localhost:8000"  # Url for debugger. change it to your own


## HF preprocessing before API

This mirrors the alignment pipeline cleaning so you can inspect exactly which text is sent to `/anonymizer/predict`.


In [ ]:
TAG_RE = re.compile(r"(?is)<[^>]+>")
SCRIPT_STYLE_RE = re.compile(r"(?is)<(script|style)[^>]*>.*?</\1>")
BREAK_TAG_RE = re.compile(r"(?is)<\s*(br|/p|/div|/li|/tr)\s*/?>")

def preprocess_hf_text(text: str, collapse_lines: bool = True) -> str:
    value = str(text or "")
    value = html.unescape(value)
    value = SCRIPT_STYLE_RE.sub(" ", value)
    value = BREAK_TAG_RE.sub("\n", value)
    value = TAG_RE.sub(" ", value)
    value = value.replace("\r\n", "\n").replace("\r", "\n")
    value = "".join(
        ch for ch in value
        if ch in {"\n", "\t", " "} or not unicodedata.category(ch).startswith("C")
    )
    value = re.sub(r"[ \t\f\v]+", " ", value)
    value = re.sub(r"\n{2,}", "\n", value)
    value = "\n".join(line.strip() for line in value.split("\n"))
    value = "\n".join(line for line in value.split("\n") if line)
    value = value.strip()
    if collapse_lines:
        value = re.sub(r"\s+", " ", value).strip()
    return value

for sample in train_ds[:20]:
    clean_sample = preprocess_hf_text(sample, collapse_lines=True)
    print("RAW:", repr(sample[:220]))
    print("CLEAN:", repr(clean_sample[:220]))
    print("API payload:", {"text": clean_sample},"\n")


## Inference


In [ ]:
# Function to make single inference using the API
def get_predictions(sample: str, collapse_lines: bool = True) -> dict:
    prepared = preprocess_hf_text(sample, collapse_lines=collapse_lines)
    response = requests.post(
        url=f"{API_URL}/anonymizer/predict",
        json={"text": prepared},
        params={"use_cache": False}
    )
    response.raise_for_status()
    return response.json()


In [ ]:
predictions = get_predictions(preprocess_hf_text(train_ds[0], collapse_lines=True), collapse_lines=True)


In [ ]:
predictions

In [ ]:
len(dev_ds)

In [ ]:
raw_text = train_ds[0]
prepared_text = preprocess_hf_text(raw_text, collapse_lines=True)
raw_text, prepared_text


In [ ]:
from pathlib import Path
import json

OUT_PATH = Path('../../../resources/data/experiments/ner_langextract_alignment/hf_pii_masking_400k_es_train.jsonl')
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

def _to_py_scalar(value):
    return value.item() if hasattr(value, 'item') else value

train_rows = ds['train'].filter(lambda x: x['language'] == 'es')

written = 0
with OUT_PATH.open('w', encoding='utf-8') as f:
    for idx, row in enumerate(train_rows):
        text = preprocess_hf_text(row.get('source_text') or '', collapse_lines=True)
        if not text:
            continue

        uid = _to_py_scalar(row.get('uid'))
        record = {
            'sample_id': str(uid) if uid is not None else f'hf-train-{idx}',
            'document_id': 'ai4privacy/pii-masking-400k',
            'paragraph_id': str(idx),
            'source_path': 'hf://ai4privacy/pii-masking-400k/train',
            'text': text,
        }

        f.write(json.dumps(record, ensure_ascii=False) + '\n')
        written += 1

print(f'Wrote {written} records to: {OUT_PATH.resolve()}')
